# 02 · Degradación sintética para inpainting

**Qué hace:** genera y valida el dataset de tríos (degradada, GT, máscara) para el fine-tuning de LaMa, a partir de imágenes limpias de época; comprueba el invariante «fuera de la máscara, degradada == GT».

**Qué necesita:**
- Paquete `synthetic_degradation/` (viene en el repo, se instala con `preparar_repo()`)
- Kaggle: `shrutimandaokar2301/vintage-degraded-image-synthetic-real` (carpeta `01_Clean_Candidates_GT`)

**Qué deja escrito:** el dataset en `ROOT/_out/lama_synthetic/{train,val,test}/{images,gt,masks}` (o en Drive si se activa `montar_drive_opcional()`)

**Arranque:** primera celda de código.

In [ ]:
!pip -q install kagglehub matplotlib
from colab_setup import aplicar_parches, preparar_repo, descargar_datos, montar_drive_opcional
aplicar_parches()
ROOT = preparar_repo()
DATA = descargar_datos("kaggle:vintage-degraded", root=ROOT)

In [ ]:
!cd {ROOT} && python -m pytest -q

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from synthetic_degradation import (
    DamageConfig, make_training_sample, mask_coverage, save_sample, generate_dataset,
)

SEED = 1234
cfg = DamageConfig.vintage_base()
print("Config cargada:", cfg)

In [ ]:
# Imágenes limpias = 01_Clean_Candidates_GT del dataset de Kaggle.
# Si la descarga falla, se generan imágenes sintéticas para humo.
def load_clean_images(limit=None) -> list[np.ndarray]:
    clean_dir = None
    try:
        root = DATA["kaggle:vintage-degraded"]
        hits = list(root.rglob("01_Clean_Candidates_GT"))
        clean_dir = hits[0] if hits else None
    except Exception as e:
        print("dataset no disponible o falló:", e)

    paths = []
    if clean_dir and clean_dir.exists():
        for ext in ("*.png", "*.jpg", "*.jpeg"):
            paths.extend(sorted(clean_dir.glob(ext)))
    if not paths:
        print("Sin imágenes limpias; usando sintéticas de prueba.")
        imgs, rng = [], np.random.default_rng(0)
        n = limit if limit is not None else 6
        for _ in range(n):
            base = int(rng.integers(90, 180))
            grad = np.linspace(base - 30, base + 30, 256).astype(np.uint8)
            band = np.repeat(grad[None, :], 256, axis=0)
            imgs.append(np.stack([band, band, band], axis=-1))
        return imgs
    if limit is not None:
        paths = paths[:limit]
    return [np.array(Image.open(p).convert("RGB")) for p in paths]

clean_images = load_clean_images()
print(f"{len(clean_images)} imágenes limpias cargadas.")

In [ ]:
def validate_sample(deg, gt, mask):
    assert set(np.unique(mask)).issubset({0, 255}), "máscara no binaria"
    assert np.array_equal(deg[mask == 0], gt[mask == 0]), "degradada != gt fuera de máscara"
    cov = mask_coverage(mask)
    assert cov < 0.6, f"cobertura sospechosamente alta: {cov:.2%}"
    return cov

rng = np.random.default_rng(SEED)
coverages = []
for i, clean in enumerate(clean_images):
    deg, gt, mask = make_training_sample(clean, rng, cfg)
    cov = validate_sample(deg, gt, mask)
    coverages.append(cov)
print("OK — todas las muestras pasan los asserts.")
print("Cobertura de daño (min/media/max): "
      f"{min(coverages):.2%} / {np.mean(coverages):.2%} / {max(coverages):.2%}")

In [ ]:
n = min(3, len(clean_images))
rng = np.random.default_rng(SEED)  # re-siembra a propósito: muestra las mismas filas 0..n-1 que la validación
fig, axes = plt.subplots(n, 4, figsize=(16, 4 * n))
if n == 1:
    axes = axes[None, :]
for i in range(n):
    deg, gt, mask = make_training_sample(clean_images[i], rng, cfg)
    overlay = deg.copy()
    overlay[mask > 127] = [255, 0, 0]
    for ax, im, title in zip(
        axes[i],
        [deg, mask, gt, overlay],
        ["Degradada", "Máscara", "GT (limpia, sin daño local)", "Overlay"],
    ):
        ax.imshow(im, cmap="gray" if im.ndim == 2 else None)
        ax.set_title(title)
        ax.axis("off")
plt.tight_layout()
plt.show()
print("Verifica: el rojo (máscara) cubre SOLO el daño local, no medias regiones.")

In [ ]:
SAVE_TO_DISK = True                 # ponlo a True para escribir el dataset definitivo
VARIANTS_PER_IMAGE = 5               # nº de degradaciones por imagen limpia
OUT_DIR = ROOT / "_out" / "lama_synthetic"
# Para persistir en tu Google Drive en vez del entorno efímero: DRIVE = montar_drive_opcional(); OUT_DIR = DRIVE / "data" / "lama_synthetic"

if SAVE_TO_DISK:
    counts, skipped = generate_dataset(
        clean_images, OUT_DIR, cfg,
        seed=SEED, variants_per_image=VARIANTS_PER_IMAGE)
    total = sum(counts.values())
    print(f"Generadas {total} muestras "
          f"(train={counts['train']}, val={counts['val']}, test={counts['test']}), "
          f"descartadas {skipped}. Salida: {OUT_DIR}")
else:
    print("SAVE_TO_DISK=False — solo validación, no se escribió nada.")